In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchsummary', 'matplotlib', 'tqdm', 'torchvision'])
    print('Done.')

## 1) Set project directory

Set `PROJECT_DIR` to the folder containing `train.py`, `eval.py`, `diffusion/`, and `models/`.

If your project is on Google Drive, mount Drive first.

In [ ]:
# Optional: mount Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

In [ ]:
# EDIT THIS PATH for your Colab setup
# Example if project folder is in Drive:
# PROJECT_DIR = Path('/content/drive/MyDrive/PA1')
PROJECT_DIR = Path('/content/drive/MyDrive/PA1')

required = ['train.py', 'eval.py', 'diffusion', 'models']
missing = [name for name in required if not (PROJECT_DIR / name).exists()]
print('Project dir:', PROJECT_DIR)
if missing:
    raise FileNotFoundError(f'Missing required project items in {PROJECT_DIR}: {missing}')

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())

## 2) Configure training run

In [ ]:
from pathlib import Path

RUN_NAME = 'colab_linear'
OUT_DIR = Path('outputs') / RUN_NAME
DATA_DIR = Path('data')

TRAIN_STEPS = 100000      # adjust for budget
BATCH_SIZE = 64           # adjust for GPU memory
NUM_WORKERS = 2
SCHEDULE_TYPE = 'linear'  # 'linear' or 'cosine'
LR = 2e-4
WEIGHT_DECAY = 0.0
SAMPLE_EVERY = 5000
CKPT_EVERY = 5000

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT_DIR)

In [ ]:
import torch
device_arg = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Detected device for training:', device_arg)

In [ ]:
train_cmd = [
    sys.executable, 'train.py',
    '--data-dir', str(DATA_DIR),
    '--out-dir', str(OUT_DIR),
    '--steps', str(TRAIN_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--schedule-type', str(SCHEDULE_TYPE),
    '--lr', str(LR),
    '--weight-decay', str(WEIGHT_DECAY),
    '--sample-every', str(SAMPLE_EVERY),
    '--ckpt-every', str(CKPT_EVERY),
    '--device', str(device_arg),
]

print('Running training command:')
print(' '.join(train_cmd))
subprocess.check_call(train_cmd)

## 3) Run evaluation and sampling

In [ ]:
ckpts = sorted((OUT_DIR / 'checkpoints').glob('ddpm_step_*.pt'))
if not ckpts:
    raise FileNotFoundError(f'No checkpoints found in {OUT_DIR / "checkpoints"}')
latest_ckpt = ckpts[-1]
print('Latest checkpoint:', latest_ckpt)

In [ ]:
EVAL_OUT = OUT_DIR / 'eval'

eval_cmd = [
    sys.executable, 'eval.py',
    '--checkpoint', str(latest_ckpt),
    '--out-dir', str(EVAL_OUT),
    '--samples', '64',
    '--sampling-steps-ablation', '1000,250,100,50',
    '--run-nn-check',
    '--data-dir', str(DATA_DIR),
    '--device', str(device_arg),
    '--train-log', str(OUT_DIR / 'train_log.csv'),
]

print('Running evaluation command:')
print(' '.join(eval_cmd))
subprocess.check_call(eval_cmd)

## 4) Preview outputs

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

preview_files = [
    EVAL_OUT / 'sample_grid_final.png',
    EVAL_OUT / 'fixed_seed_denoising_trajectory.png',
    EVAL_OUT / 'schedule_ablation_alpha_snr.png',
    EVAL_OUT / 'nearest_neighbor_pairs_pixel_l2.png',
]

for p in preview_files:
    print('Exists:', p.exists(), '|', p)
    if p.exists():
        img = Image.open(p)
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.title(p.name)
        plt.axis('off')
        plt.show()

In [ ]:
diag_file = EVAL_OUT / 'diagnostics.txt'
if diag_file.exists():
    print(diag_file.read_text())
else:
    print('No diagnostics.txt found yet.')